# 13 — Encode Shaoxing with verified original ECG-HuBERT

Experiment 4 consumes saved bridge features. Their original encoder is in notebook2/03_encode_ecg_hubert.ipynb. This notebook locates the cached checkpoint from the bridge config and numerically checks ten frozen training-bank embeddings before encoding. Use the ORIGINAL HuBERT CUDA environment with torch, transformers, numpy and wfdb. No training, checkpoint download or manual equivalence override.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'config.json').is_file()), None)
if ROOT is None: raise FileNotFoundError('Start inside graph_reasoning')
sys.path.insert(0, str(ROOT))
from report_study import read, resolve
CONFIG = read(ROOT / 'config.json')
OUT = resolve(ROOT, CONFIG['output_dir'])
BASE_STUDY = OUT / 'report_study_v1'
EXTERNAL = OUT / 'shaoxing_external_v1'
print('Kernel:', sys.executable)
print('External cohort:', EXTERNAL)


In [ ]:
# CPU-only preflight, before model loading. Rerun to resume a timed-out audit.
from shaoxing_waveform_audit import audit
waveform_audit = audit(EXTERNAL, max_run_hours=1.0)
print({k:v for k,v in waveform_audit.items() if k != 'failures'})
for failure in waveform_audit['failures'][:10]: print(failure)
print('Full audit:', EXTERNAL / 'waveform_validation_audit.json')
if not waveform_audit['complete']:
    raise RuntimeError('Waveform audit incomplete; rerun this cell before proceeding.')
if waveform_audit['invalid']:
    raise ValueError('Invalid frozen-cohort ECGs found. Review waveform_validation_audit.json; no records were removed or imputed.')


In [ ]:
from shaoxing_hubert_adapter import setup, verify_bank, DESCRIPTION
from report_study import sha
ARTIFACT_ROOT = Path('/path/to/data/symile/ecg_cxr_bridge_artifacts')
SNAPSHOT_PATH = None  # Auto-discover from original bridge config hf_cache; set exact snapshot if ambiguous.
ADAPTER_PATH = ROOT / 'shaoxing_hubert_adapter.py'
CHECKPOINT_PATH = setup(ARTIFACT_ROOT, EXTERNAL, snapshot=SNAPSHOT_PATH)
print('Encoder:', read(CHECKPOINT_PATH)['model_id'])
print('Snapshot:', read(CHECKPOINT_PATH)['snapshot'])
print('Checkpoint fingerprint manifest:', CHECKPOINT_PATH)


In [ ]:
manifest = resolve(ROOT, CONFIG['export_sources']['manifest'])
SENSOR_SOURCES = manifest.with_name('sensor_sources.json')
SYMILE_ROOT = resolve(ROOT, CONFIG['symile_root'])
print('Verification tensors:', SYMILE_ROOT / 'data_npy/train/ecg_train.npy')
verification = verify_bank(OUT, SENSOR_SOURCES, EXTERNAL, probes=10, symile_root=SYMILE_ROOT)
print(verification)


In [ ]:
from shaoxing_external import encode
PREPROCESSING_DESCRIPTION = DESCRIPTION
BANK_EQUIVALENCE_CONFIRMED = verification['passed']
MAX_RUN_HOURS = 30.0  # Below remaining allocation time after loading and verification.
provenance = dict(checkpoint_path=str(CHECKPOINT_PATH), checkpoint_sha256=sha(CHECKPOINT_PATH),
                  preprocessing_description=PREPROCESSING_DESCRIPTION,
                  bank_equivalence_confirmed=BANK_EQUIVALENCE_CONFIRMED,
                  bank_equivalence_audit=verification)
print(encode(EXTERNAL, OUT, ADAPTER_PATH, provenance, max_run_hours=MAX_RUN_HOURS))
